In [32]:
pip install torch transformers datasets "lm_eval[hf]" ipykernel

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = str(NUM_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:

import json
import platform
import subprocess
import time
from datetime import datetime
 
import torch
import transformers
import lm_eval
from lm_eval.models.huggingface import HFLM
from transformers import AutoModelForCausalLM, AutoTokenizer
torch.set_num_threads(NUM_THREADS)

1


In [ ]:

def get_machine_info():
    """Record which machine and software this run used."""
    info = {
        "system": platform.system(),        # Darwin = macOS, Linux = Colab/Kaggle
        "machine": platform.machine(),      # arm64 = Apple Silicon
        "cpu_count": os.cpu_count(),
        "torch_threads": torch.get_num_threads(),
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
    }
    if info["system"] == "Darwin":
        info["cpu_name"] = subprocess.run(
            ["sysctl", "-n", "machdep.cpu.brand_string"],
            capture_output=True, text=True,
        ).stdout.strip()
    else:
        info["cpu_name"] = platform.processor()
    return info

In [ ]:

def load_model(model_name, device, dtype_name):
    """Load tokenizer and model on the chosen device and precision."""
    dtype = getattr(torch, dtype_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, dtype=dtype, device_map=device
    )
    model.eval()
    return model, tokenizer

In [ ]:

 
def check_setup(model, expected_device, expected_dtype_name, expected_threads):
    """Stop the run early if the setup is not what the experiment claims."""
    actual_device = model.device.type
    actual_dtype = str(model.dtype).replace("torch.", "")
    actual_threads = torch.get_num_threads()
    print(f"device={actual_device}  dtype={actual_dtype}  threads={actual_threads}")
    assert actual_device == expected_device, f"Model is on {actual_device}, expected {expected_device}"
    assert actual_dtype == expected_dtype_name, f"Model dtype is {actual_dtype}, expected {expected_dtype_name}"
    assert actual_threads == expected_threads, f"Torch threads is {actual_threads}, expected {expected_threads}"

In [ ]:

def run_accuracy(model, tokenizer, tasks, batch_size, limit):
    """Accuracy check with lm_eval. Returns results and total wall-clock time.
 
    Note: passing a model object means lm_eval ignores its own device
    argument, so where the model was loaded is what counts.
    """
    lm = HFLM(pretrained=model, tokenizer=tokenizer, batch_size=batch_size)
    start = time.perf_counter()
    results = lm_eval.simple_evaluate(model=lm, tasks=tasks, limit=limit)
    elapsed = time.perf_counter() - start
    return results, elapsed
 

In [ ]:

def summarise(results, tasks):
    """Pull out just the metrics for each task."""
    return {task: results["results"][task] for task in tasks}
 
 
def save_results(path, settings, machine_info, summary, elapsed):
    """Append one run to a JSON-lines file so earlier runs are never lost."""
    record = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "settings": settings,
        "machine": machine_info,
        "elapsed_seconds": round(elapsed, 3),
        "metrics": summary,
    }
    with open(path, "a") as f:
        f.write(json.dumps(record, default=str) + "\n")
 

In [ ]:

def run_timing_loop(model, tokenizer, questions, max_new_tokens, out_csv):
    """YOUR PART: per-question timing for RQ1.
 
    For each question it should:
      1. build the prompt (chat template for Instruct models)
      2. tokenise it and count input tokens
      3. time generate() with do_sample=False and fixed max_new_tokens,
         inside torch.inference_mode(), using time.perf_counter()
      4. decode only the new tokens and count them
      5. write one row to out_csv immediately (so a crash loses nothing)
    Skip the first one or two questions as warm-up when you analyse.
    """
    raise NotImplementedError
 

In [ ]:
NUM_THREADS = 1
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cpu"
DTYPE_NAME = "float32"
BATCH_SIZE = 1
TASKS = ["arc_challenge"]
LIMIT = 50                      # questions per task; None = full dataset
RESULTS_FILE = "results_accuracy.json"

In [ ]:
def main():
    settings = {
        "model": MODEL_NAME, "device": DEVICE, "dtype": DTYPE_NAME,
        "threads": NUM_THREADS, "batch_size": BATCH_SIZE,
        "tasks": TASKS, "limit": LIMIT,
    }
    machine_info = get_machine_info()
    print(machine_info)
 
    model, tokenizer = load_model(MODEL_NAME, DEVICE, DTYPE_NAME)
    check_setup(model, DEVICE, DTYPE_NAME, NUM_THREADS)
 
    results, elapsed = run_accuracy(model, tokenizer, TASKS, BATCH_SIZE, LIMIT)
    summary = summarise(results, TASKS)
    print(summary)
    print(f"Total time: {elapsed:.1f} s")
 
    save_results(RESULTS_FILE, settings, machine_info, summary, elapsed)
 
 
if __name__ == "__main__":
    main()
 